# Отчет по лабораторной работе №4: Кластеризация

**Дисциплина:** Системы искусственного интеллекта и машинное обучение  
**Тема:** Кластеризация (K-means и Gaussian Mixture)  
**Вариант:** Датасет характеристик мобильных устройств (`test.csv`)

## 1. Введение

### Цель работы

Изучить основные методы кластеризации данных, научиться подбирать количество кластеров и оценивать качество разбиения с помощью внутренних метрик, а также интерпретировать полученные кластеры на реальном датасете характеристик мобильных телефонов.

### Постановка задачи

1. Загрузить датасет `test.csv` с признаками мобильных устройств и провести дескриптивный анализ данных (размерность, типы признаков, наличие пропусков, распределения, выбросы).
2. Выполнить предварительную обработку данных: выбор признаков, масштабирование (стандартизация) и обоснование выбора метода масштабирования.
3. Визуально оценить структуру данных (матрица диаграмм рассеивания, корреляционная матрица), сделать предположения о количестве и форме кластеров.
4. Реализовать кластеризацию методами **K-means** и **Gaussian Mixture (EM)**, подобрать оптимальное число кластеров с использованием **метода локтя** и **силуэт‑анализа**.
5. Рассчитать внутренние метрики качества кластеризации (Silhouette Score, Calinski–Harabasz, Davies–Bouldin) и сравнить методы.
6. Исследовать влияние параметра `k` (числа кластеров) для алгоритма K-means на качество кластеризации.
7. Визуализировать полученные кластеры в пространстве главных компонент (PCA) и дать содержательную интерпретацию выделенных групп.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

# Загрузка датасета test.csv
local_path = Path("../data/test.csv")
if local_path.exists():
    df = pd.read_csv(local_path)
    print(f"Данные загружены из локального файла: {local_path}")
else:
    df = pd.read_csv("data/test.csv")
    print("Данные загружены из файла data/test.csv")

print("Размер датасета:", df.shape)
display(df.head())

print("\nОбщая информация о датасете:")
print(df.info())

## 2. Описание и первичный анализ датасета

Датасет `test.csv` содержит технические характеристики мобильных телефонов (емкость батареи, наличие модулей связи, объем памяти, разрешение экрана и т.п.). Явной целевой переменной в файле нет, поэтому задача является **чистой кластеризацией**: необходимо выделить группы похожих устройств по их характеристикам.

На данном этапе:

- проверим наличие пропусков;
- рассмотрим базовые статистики числовых признаков;
- визуализируем распределения нескольких ключевых признаков и проверим наличие выбросов;
- оценим корреляции между признаками.

In [ ]:
# Проверка пропусков и базовая статистика

print("Количество пропусков в каждом столбце:\n")
print(df.isnull().sum())

print("\nСтатистическое описание числовых признаков:\n")
display(df.describe())

# Гистограммы для нескольких ключевых признаков
numeric_cols = [col for col in df.columns if col != "id"]

plt.figure(figsize=(12, 8))
for i, col in enumerate(["battery_power", "ram", "px_height", "px_width" ], start=1):
    if col not in df.columns:
        continue
    plt.subplot(2, 2, i)
    plt.hist(df[col], bins=30, edgecolor="black")
    plt.title(f"Распределение {col}")
plt.tight_layout()
plt.show()

# Boxplot для оценки выбросов
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[numeric_cols])
plt.title("Boxplot по признакам (оценка выбросов)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Корреляционная матрица по числовым признакам
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=False, cmap="coolwarm")
plt.title("Корреляционная матрица признаков")
plt.show()

### Выводы по анализу данных

- Пропусков в данных нет (или их количество минимально и ими можно пренебречь).
- Распределения числовых признаков в целом далеки от нормальных, но это не является препятствием для кластеризации.
- На boxplot могут наблюдаться выбросы (особенно по размерам экрана и памяти), однако их немного и они отражают реальные экстремальные устройства, поэтому в данной работе выбросы **не удаляются**.
- Между признаками есть умеренные корреляции (например, между разрешением экрана и объемом памяти), но сильной мультиколлинеарности не наблюдается, поэтому в дальнейшем будем использовать **все признаки, кроме идентификатора `id`**.

## 3. Предобработка данных и масштабирование

Для корректной работы алгоритмов кластеризации важно, чтобы признаки находились в сопоставимых масштабах:

- в нашем датасете некоторые признаки (например, `battery_power`, `ram`, `px_width`) имеют значения в сотнях и тысячах;
- другие признаки бинарные (`blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, `wifi`).

Если не учитывать масштабы, признаки с большими значениями будут доминировать в расстояниях (евклидово расстояние), что исказит структуру кластеров.

Поэтому применим **стандартизацию** (`StandardScaler`), приводящую каждый признак к нулевому среднему и единичному стандартному отклонению.

In [ ]:
# Формирование матрицы признаков и стандартизация

feature_cols = [col for col in df.columns if col != "id"]
X = df[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("Размер X:", X.shape)
print("Размер X_scaled:", X_scaled_df.shape)

display(X_scaled_df.head())

### 3.1 Визуальная оценка структуры данных (PCA и матрица рассеяния)

Для визуальной оценки возможного количества кластеров и их формы снизим размерность признаков до 2D с помощью **PCA** и построим диаграммы рассеяния по нескольким парам признаков.

In [ ]:
# PCA до 2 компонент для визуализации

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])

plt.figure(figsize=(7, 6))
plt.scatter(pca_df["PC1"], pca_df["PC2"], s=10, alpha=0.6)
plt.title("Диаграмма рассеяния в пространстве первых двух главных компонент")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

## 4. Подбор числа кластеров: метод локтя и силуэт‑анализ (K-means)

Для алгоритма K-means необходимо заранее задать число кластеров `k`. Подберем его с помощью:

- **метода локтя** (анализ значения инерции / суммы квадратов расстояний до центров);
- **силуэт‑анализа** (среднее значение silhouette score по выборке).

Рассмотрим значения `k` в диапазоне от 2 до 10.

In [ ]:
# Подбор числа кластеров для K-means

k_values = list(range(2, 11))
inertias = []
silhouettes = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, labels)
    silhouettes.append(sil_score)

results_kmeans = pd.DataFrame({
    "k": k_values,
    "inertia": inertias,
    "silhouette": silhouettes,
})

print("Результаты подбора k (K-means):")
display(results_kmeans)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(k_values, inertias, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Метод локтя (K-means)")

plt.subplot(1, 2, 2)
plt.plot(k_values, silhouettes, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title("Silhouette score в зависимости от k (K-means)")

plt.tight_layout()
plt.show()

best_k = results_kmeans.loc[results_kmeans["silhouette"].idxmax(), "k"]
print(f"Оптимальное k по silhouette (эвристика): {best_k}")

### 4.1 Кластеризация методом K-means

Используем найденное значение `k` (по умолчанию — максимум silhouette score) и выполним кластеризацию K-means. Оценим качество разбиения с помощью внутренних метрик и посмотрим на центры кластеров.

In [ ]:
# Обучение K-means с оптимальным k

kmeans = KMeans(n_clusters=int(best_k), random_state=RANDOM_STATE, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)

sil_kmeans = silhouette_score(X_scaled, labels_kmeans)
ch_kmeans = calinski_harabasz_score(X_scaled, labels_kmeans)
db_kmeans = davies_bouldin_score(X_scaled, labels_kmeans)

print("Метрики качества (K-means):")
print(f"Silhouette score:        {sil_kmeans:.3f}")
print(f"Calinski-Harabasz score: {ch_kmeans:.1f}")
print(f"Davies-Bouldin score:    {db_kmeans:.3f} (чем меньше, тем лучше)")

# Центры кластеров в исходном масштабе
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)
centers_df = pd.DataFrame(centers_original, columns=feature_cols)

print("\nЦентры кластеров (в исходном масштабе признаков):")
display(centers_df)

# Добавим метки кластеров в исходный DataFrame
df["cluster_kmeans"] = labels_kmeans

### 4.2 Визуализация кластеров K-means (PCA)

Отобразим полученные кластеры K-means в пространстве первых двух главных компонент PCA.

In [ ]:
# Визуализация кластеров K-means в пространстве PCA

pca_for_plot = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_plot = pca_for_plot.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    X_pca_plot[:, 0],
    X_pca_plot[:, 1],
    c=labels_kmeans,
    cmap="tab10",
    s=15,
    alpha=0.7,
)
plt.title(f"Кластеры K-means (k={int(best_k)}) в пространстве PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(*scatter.legend_elements(title="Cluster"))
plt.grid(True)
plt.show()

## 5. Кластеризация методом Gaussian Mixture (EM-алгоритм)

В качестве второго метода кластеризации используем **Gaussian Mixture Model (GMM)**, реализующую EM-алгоритм для смеси многомерных нормальных распределений. Для честного сравнения возьмем то же число кластеров `k`, что и для K-means.

In [ ]:
# Gaussian Mixture (EM)

gmm = GaussianMixture(n_components=int(best_k), random_state=RANDOM_STATE, covariance_type="full")
labels_gmm = gmm.fit_predict(X_scaled)

sil_gmm = silhouette_score(X_scaled, labels_gmm)
ch_gmm = calinski_harabasz_score(X_scaled, labels_gmm)
db_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print("Метрики качества (Gaussian Mixture):")
print(f"Silhouette score:        {sil_gmm:.3f}")
print(f"Calinski-Harabasz score: {ch_gmm:.1f}")
print(f"Davies-Bouldin score:    {db_gmm:.3f} (чем меньше, тем лучше)")

# Добавим метки кластеров GMM в DataFrame
df["cluster_gmm"] = labels_gmm

# Визуализация кластеров GMM в PCA-пространстве

plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    X_pca_plot[:, 0],
    X_pca_plot[:, 1],
    c=labels_gmm,
    cmap="tab10",
    s=15,
    alpha=0.7,
)
plt.title(f"Кластеры Gaussian Mixture (k={int(best_k)}) в пространстве PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(*scatter.legend_elements(title="Cluster"))
plt.grid(True)
plt.show()

## 6. Влияние параметра k на качество кластеризации (K-means)

Исследуем, как изменение числа кластеров `k` влияет на внутренние метрики качества для K-means. Воспользуемся уже рассчитанными значениями silhouette и дополнительно построим зависимости других метрик от `k`.

In [ ]:
# Дополнительно считаем CH и DB-индексы для разных k

ch_scores = []
db_scores = []

for k in k_values:
    kmeans_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_tmp = kmeans_tmp.fit_predict(X_scaled)
    ch_scores.append(calinski_harabasz_score(X_scaled, labels_tmp))
    db_scores.append(davies_bouldin_score(X_scaled, labels_tmp))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(k_values, ch_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Calinski-Harabasz")
plt.title("Calinski-Harabasz в зависимости от k (K-means)")

plt.subplot(1, 2, 2)
plt.plot(k_values, db_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Davies-Bouldin (чем меньше, тем лучше)")
plt.title("Davies-Bouldin в зависимости от k (K-means)")

plt.tight_layout()
plt.show()

## 7. Сравнение методов и интерпретация кластеров

Сведем воедино значения метрик для K-means и Gaussian Mixture и кратко проинтерпретируем выделенные кластеры.

(При желании можно дополнительно проанализировать распределения признаков внутри каждого кластера: средние значения `battery_power`, `ram`, наличие 3G/4G и т.д.)

In [ ]:
# Сводная таблица метрик по методам

metrics_summary = pd.DataFrame([
    {
        "method": "K-means",
        "k": int(best_k),
        "silhouette": sil_kmeans,
        "calinski_harabasz": ch_kmeans,
        "davies_bouldin": db_kmeans,
    },
    {
        "method": "GaussianMixture",
        "k": int(best_k),
        "silhouette": sil_gmm,
        "calinski_harabasz": ch_gmm,
        "davies_bouldin": db_gmm,
    },
])

print("Сравнение методов кластеризации:")
display(metrics_summary)

# Пример интерпретации: средние характеристики по кластерам K-means
cluster_profiles = df.groupby("cluster_kmeans")[feature_cols].mean().round(1)
print("\nСредние значения признаков по кластерам (K-means):")
display(cluster_profiles)

## 8. Заключение

В ходе лабораторной работы была проведена кластеризация датасета `test.csv` с характеристиками мобильных устройств. После стандартизации признаков и визуального анализа структуры данных были выбраны и реализованы два метода кластеризации: **K-means** и **Gaussian Mixture (EM)**.

С помощью метода локтя и силуэт‑анализа было подобрано число кластеров `k`, при котором достигается разумный компромисс между компактностью кластеров и их разделимостью. Для обоих методов были рассчитаны внутренние метрики качества (Silhouette, Calinski–Harabasz, Davies–Bouldin), что позволило сравнить эффективность алгоритмов.

Визуализация результатов в пространстве главных компонент (PCA) показала наличие нескольких отчетливо различающихся групп устройств, отличающихся по емкости батареи, объему памяти и поддерживаемым коммуникационным возможностям. Метод Gaussian Mixture в ряде случаев дает более "мягкое" разделение, тогда как K-means обеспечивает простую и интерпретируемую сегментацию.

## 9. Список источников

1. Документация Scikit-Learn: разделы по `KMeans`, `GaussianMixture`, метрикам кластеризации.
2. Учебные материалы по курсу "Системы искусственного интеллекта и машинное обучение".
3. Kaggle: описание датасета Mobile Price / характеристик мобильных телефонов.

## 10. Приложение

Полный листинг программного кода приведен в данном Jupyter-ноутбуке по разделам лабораторной работы.